# e5 unseen-country adaptation (D-031): continue fine-tuning the saved multilingual-e5-small (MIT) on French
pseudo-labelled test pairs + labelled train pairs (label-free for France), then re-score every French pair.
Needs in My Drive: `ber_e5_fr` (adapt.parquet, score_test.parquet) and the `ber_ce` shortcut (model).
Runtime → Change runtime type → **T4 GPU**; Runtime → Run all. Writes `My Drive/ber_e5_fr/out/ce_test.parquet` (~25 min).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, glob, time, math, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
direct = '/content/drive/MyDrive/ber_ce/out/model'
cands = [direct + '/model.safetensors'] if os.path.exists(direct + '/model.safetensors') else \
        glob.glob('/content/drive/MyDrive/**/model/model.safetensors', recursive=True)
bund = glob.glob('/content/drive/MyDrive/**/ber_e5_fr/adapt.parquet', recursive=True)
print('models found:', cands); print('bundle found:', bund)
MODEL, BUNDLE = os.path.dirname(cands[0]), os.path.dirname(bund[0])
OUT = BUNDLE + '/out'; os.makedirs(OUT, exist_ok=True)
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU'
torch.manual_seed(42); np.random.seed(42)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL).cuda()
MAXLEN, BS, LR = 64, 128, 2e-5
tr = pd.read_parquet(f'{BUNDLE}/adapt.parquet').sample(frac=1.0, random_state=42).reset_index(drop=True)
print('adapt pairs', len(tr), 'pos rate', tr.label.mean())

In [ ]:
def batches(df, bs, shuffle, seed=0):
    idx = np.random.default_rng(seed).permutation(len(df)) if shuffle else np.arange(len(df))
    for s in range(0, len(df), bs):
        j = idx[s:s + bs]
        enc = tok(df.ta.values[j].tolist(), df.tb.values[j].tolist(), truncation=True, max_length=MAXLEN,
                  padding=True, return_tensors='pt')
        yield j, {k: v.cuda(non_blocking=True) for k, v in enc.items()}
steps = math.ceil(len(tr) / BS)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 300) * max(0.0, 1 - s / steps))
scaler = torch.amp.GradScaler('cuda')
y_all = torch.tensor(tr.label.values, dtype=torch.float32)
model.train(); t0 = time.time()
for step, (j, enc) in enumerate(batches(tr, BS, True, seed=42), 1):
    with torch.autocast('cuda', dtype=torch.float16):
        logit = model(**enc).logits.squeeze(-1)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logit.float(), y_all[j].cuda())
    scaler.scale(loss).backward(); scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
    if step % 500 == 0:
        el = time.time() - t0
        print(f'step {step}/{steps} loss {loss.item():.4f} {el/60:.1f} min, eta {el/step*(steps-step)/60:.1f} min', flush=True)
print('adapted', (time.time() - t0) / 60, 'min')

In [ ]:
model.eval()
df = pd.read_parquet(f'{BUNDLE}/score_test.parquet')
order = np.argsort((df.ta.str.len() + df.tb.str.len()).values, kind='stable')
df = df.iloc[order].reset_index(drop=True)
out = np.zeros(len(df), dtype=np.float32); t0 = time.time()
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
    for j, enc in batches(df, 1024, False):
        out[j] = model(**enc).logits.squeeze(-1).float().cpu().numpy()
pd.DataFrame({'lid': df.lid, 'rid': df.rid, 'ce': out}).to_parquet(f'{OUT}/ce_test.parquet', index=False)
print('scored', len(df), f'{(time.time()-t0)/60:.1f} min')
print('DONE — download My Drive/ber_e5_fr/out/ce_test.parquet')